# A* Path Planning Algorithm

The A* (A-star) algorithm is a widely used and highly effective pathfinding and graph traversal algorithm. It is heavily utilized in robotics for navigating 2D occupancy grids (like planning a route for a mobile robot avoiding obstacles).

A* selects the path that minimizes the total estimated cost, evaluating nodes based on the following function:
**f(n) = g(n) + h(n)**

* **g(n)**: The exact cost of the path from the starting node to the current node 'n'.
* **h(n)**: The heuristic estimated cost from node 'n' to the goal. This must be an "admissible" heuristic, meaning it never overestimates the true cost.

This notebook implements A* on a 2D grid, using the Euclidean distance as the heuristic to find the shortest path from a start position to a goal position while avoiding obstacles.

## Part 1: Node Definition and Heuristics

First, we define a `Node` class. This allows us to keep track of a node's position on our grid, its parent (so we can retrace the final path), and its specific costs (`g`, `h`, and `f`). 

We will also define a heuristic function. Since our robot can move in 8 directions (including diagonals), Euclidean distance provides a straightforward and admissible heuristic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class Node:
    #initialize the node with an optional parent node and a position
    def __init__(self,parent=None,position=None):
        self.parent=parent #the node object we came from
        self.position=position #(x,y) coordiniates

        self.g=0 #distance to start node
        self.h=0 #distance to goal node
        self.f=0 #total cost g+h


    # Define equality so we can easily check if two nodes share the same position
    def __eq__(self, other):
        return self.position==other.position

def heuristic(current_pos,goal_pos):
    #calculate the euclidean distance between two points
    #both inputs are tuples representing coordinates
    return np.sqrt((current_pos[0]-goal_pos[0])**2+(current_pos[1]-goal_pos[1])**2)

    

## Part 2: The A* Search Algorithm

The core logic maintains two lists:
1. **Open List**: Nodes that have been discovered but not yet evaluated.
2. **Closed List**: Nodes that have already been evaluated.

The algorithm loops by continually pulling the node with the lowest `f` cost from the open list, generating its neighbors, and evaluating them. If a neighbor is the goal, we trace the `parent` pointers back to the start to get the final path.

In [ ]:
def astar(maze,start,goal):
    #maze is a 2d numpy array where 0 is a free space and 1 is an obstacle
    #start:tuple of the start pos, end: tuple of the end pos

    #create start and goal nodes
    start_node=Node(None,start)
    start_node.g=start_node.h=start_node.f=0
    goal_node=Node(None,goal)
    goal_node.g=goal_node.h=goal.f=0

    #initialize open and closed lists
    open_list=[] # list of node objects
    closed_list=[] #list of node objs

    #add the start node to the open list
    open_list.append(start_node)

    #loop till the open list is empty or we find the goal
    while len(open_list)>0:

        #get the current node with the lowest f cost
        current_node=open_list[0]
        current_index=0
        for index,item in enumerate(open_list):
            if item.f<current_node.f:
                current_node=item
                current_index=index
    
        #pop current off open list, and add it to closed list
        open_list.pop(current_index)
        closed_list.append(current_node)

        #check if we found the goal
        if current_node==goal_node:
            path=[]
            current=current_node
            #retrace the path backwards
            while current is not None:
                path.append(current.position)
                current=current.parent

            return path[::-1]#reverse path so that it goes from start to goal
        
        #generate children/adjacent nodes
        children=[]

        # These 8 tuples represent the relative movements: up, down, left, right, and the 4 diagonals
        for new_position in [(0,-1),(0,1),(-1,0),(1,0),(-1,-1),(-1.1),(1,1),(1,-1)]:

            #get the exact node position
            node_position=(current_node[0]+new_position[0],current_node[1]+new_position[1])
            # Make sure the new position is within the bounds of the maze (e.g., inside the 10x10 grid)
            if node_position[0] > (len(maze) - 1) or node_position[0] < 0 or node_position[1] > (len(maze[len(maze)-1]) -1) or node_position[1] < 0:
                continue

            # Make sure the new position is walkable terrain (0 = free, 1 = obstacle)
            if maze[node_position[0]][node_position[1]] != 0:
                continue

            new_node=Node(current_node,node_position)
            children.append(new_node)



